# Clasificacion de Colegios Bilingues - Departamento del Cesar

**Proyecto 2 - Analitica Computacional**

**Pregunta de Negocio**: Es posible predecir si un colegio es bilingue
basandose en el desempeno de sus estudiantes en el Saber 11 y
caracteristicas institucionales?

**Datos**: Reales del Cesar (77,575 registros)

**Cliente**: Ministerio de Educacion Nacional


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print('Librerias importadas')


In [ ]:
# Cargar datos REALES del Cesar
df = pd.read_csv("../data/clean_icfes_data_cesar.csv")
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

print(f"Dataset cargado: {df.shape[0]:,} registros, {df.shape[1]} columnas")
df.head()


In [ ]:
# Distribucion de la variable objetivo
print('Distribucion de cole_bilingue:')
print(df['cole_bilingue'].value_counts())
print('\nPorcentajes:')
print(df['cole_bilingue'].value_counts(normalize=True) * 100)

plt.figure(figsize=(8, 5))
df['cole_bilingue'].value_counts().plot(kind='bar', color=['steelblue', 'darkorange'])
plt.title('Colegios Bilingues vs No Bilingues', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de Colegio')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Eliminar NaN en cole_bilingue
print(f"Antes: {len(df):,} registros")
print(f"NaN en cole_bilingue: {df['cole_bilingue'].isna().sum():,}")

df = df.dropna(subset=['cole_bilingue'])
print(f"Después: {len(df):,} registros")

# Normalizar nombres de columnas
df.columns = df.columns.str.lower()


In [ ]:
# Comparar puntajes promedio por tipo de colegio
puntajes = ['punt_ingles', 'punt_matematicas', 'punt_lectura_critica', 'punt_global']

print('Puntajes promedio por tipo de colegio:')
comparacion = df.groupby('cole_bilingue')[puntajes].mean()
print(comparacion.round(1))

diff = comparacion.loc['S', 'punt_ingles'] - comparacion.loc['N', 'punt_ingles']
print(f'  Bilingues tienen {diff:.1f} puntos mas en ingles')


In [ ]:
# Histogramas de puntajes por grupo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribucion de Puntajes: Bilingues vs No Bilingues', fontsize=16, fontweight='bold')

for idx, col in enumerate(puntajes):
    ax = axes[idx // 2, idx % 2]
    data_n = df[df['cole_bilingue'] == 'N'][col].dropna()
    data_s = df[df['cole_bilingue'] == 'S'][col].dropna()
    ax.hist(data_n, bins=30, alpha=0.6, label='No Bilingue', color='steelblue')
    ax.hist(data_s, bins=30, alpha=0.6, label='Bilingue', color='darkorange')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Frecuencia')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Features para el modelo
features_num = [
    'punt_lectura_critica', 'punt_matematicas', 'punt_sociales_ciudadanas',
    'punt_c_naturales', 'punt_ingles', 'punt_global', 'fami_estratovivienda'
]

features_cat = [
    'cole_naturaleza', 'cole_jornada', 'estu_genero',
    'fami_tieneinternet', 'fami_tienecomputador'
]

print(f"Features numéricas: {len(features_num)}")
print(f"Features categóricas: {len(features_cat)}")


In [ ]:
# Preparar dataset
all_features = features_num + features_cat + ['cole_bilingue']
df_clean = df[all_features].copy()

print(f"Antes de limpieza: {len(df_clean):,}")
df_clean = df_clean.dropna()
print(f"Después de limpieza: {len(df_clean):,}")

# Convertir estrato a numérico
df_clean['fami_estratovivienda'] = df_clean['fami_estratovivienda'].str.extract(r'(\d+)', expand=False).astype(float)
df_clean = df_clean.dropna(subset=['fami_estratovivienda'])

print(f"\nFinal: {len(df_clean):,} registros")
print(f"Bilingües: {(df_clean['cole_bilingue']=='S').sum():,} ({(df_clean['cole_bilingue']=='S').sum()/len(df_clean)*100:.2f}%)")


In [ ]:
# One-hot encoding
df_encoded = pd.get_dummies(df_clean, columns=features_cat, drop_first=True, dtype=float)

print(f"Columnas después de encoding: {df_encoded.shape[1]}")

# Listar nuevas columnas
nuevas = [col for col in df_encoded.columns if col not in features_num and col != 'cole_bilingue']
print(f"\nColumnas creadas por encoding: {len(nuevas)}")


In [ ]:
# Preparar X e y
feature_cols = [col for col in df_encoded.columns 
                if col in features_num or any(cat in col for cat in features_cat)]

X = df_encoded[feature_cols].copy()
y = (df_encoded['cole_bilingue'] == 'S').astype(int)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Bilingües: {y.sum():,} ({y.sum()/len(y)*100:.2f}%)")
print(f"\nFeatures utilizadas: {len(feature_cols)}")


In [ ]:
from sklearn.model_selection import train_test_split

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} ({y_train.sum():,} bilingües, {y_train.sum()/len(y_train)*100:.2f}%)")
print(f"Test:  {X_test.shape[0]:,} ({y_test.sum():,} bilingües, {y_test.sum()/len(y_test)*100:.2f}%)")


In [ ]:
from sklearn.utils import class_weight

# Pesos de clase para compensar el desbalance (~2% bilingues)
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}

print(f'Peso No Bilingue: {class_weight_dict[0]:.2f}')
print(f'Peso Bilingue:    {class_weight_dict[1]:.2f}')
print(f'Razon: {class_weight_dict[1]/class_weight_dict[0]:.1f}x mas peso a bilingues')


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow: {tf.__version__}')


In [ ]:
# Capa de normalizacion adaptada al set de entrenamiento
def build_norm_layer(X_arr):
    norm = tf.keras.layers.Normalization()
    norm.adapt(X_arr)
    return norm

X_train_arr = X_train.astype(float).values
X_test_arr = X_test.astype(float).values

norm_layer = build_norm_layer(X_train_arr)
print('Normalizacion creada')


In [ ]:
# Modelo baseline
model_baseline = Sequential([
    Input(shape=(X_train.shape[1],)),
    norm_layer,
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model_baseline.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

model_baseline.summary()


In [ ]:
%%time
# Entrenamiento del modelo baseline
early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)

history_baseline = model_baseline.fit(
    X_train_arr, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

print('Entrenamiento completado')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Predicciones sobre conjunto de prueba
y_pred_proba = model_baseline.predict(X_test_arr, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)

print('Resultados - Modelo Baseline')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1-Score:  {f1:.4f}')
print(f'  AUC-ROC:   {auc:.4f}')
print('\nMatriz de Confusion:')
print(confusion_matrix(y_test, y_pred))
print('\nReporte:')
print(classification_report(y_test, y_pred, target_names=['No Bilingue', 'Bilingue']))


In [ ]:
# Curvas de entrenamiento del baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_baseline.history['loss'], label='Entrenamiento')
axes[0].plot(history_baseline.history['val_loss'], label='Validacion')
axes[0].set_title('Perdida')
axes[0].set_xlabel('Epocas')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_baseline.history['auc'], label='Entrenamiento')
axes[1].plot(history_baseline.history['val_auc'], label='Validacion')
axes[1].set_title('AUC-ROC')
axes[1].set_xlabel('Epocas')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import mlflow
import mlflow.keras
import os

# Configurar MLflow con tracking local
os.environ['MLFLOW_TRACKING_URI'] = 'file:./mlruns'
mlflow.set_experiment('Clasificacion_Colegios_Bilingues_CESAR')

print('MLflow configurado')
print(f'  Tracking URI: {mlflow.get_tracking_uri()}')


In [ ]:
# Definir arquitecturas a probar
ARCHITECTURES = {
    'small': {'layers': [64, 32], 'dropouts': [0.3, 0.2]},
    'medium': {'layers': [128, 64, 32], 'dropouts': [0.4, 0.3, 0.2]},
    'large': {'layers': [256, 128, 64, 32], 'dropouts': [0.5, 0.4, 0.3, 0.2]},
    'deep': {'layers': [128, 64, 32, 16], 'dropouts': [0.4, 0.3, 0.2, 0.1]}
}

for name, config in ARCHITECTURES.items():
    print(f"{name:10s}: {config['layers']}")


In [ ]:
def build_model(arch, norm, n_feat, use_bn=False):
    """Construye un modelo secuencial con la arquitectura indicada."""
    layers = [Input(shape=(n_feat,)), norm]
    for units, dropout in zip(arch['layers'], arch['dropouts']):
        layers.append(Dense(units, activation='relu'))
        if use_bn:
            layers.append(BatchNormalization())
        layers.append(Dropout(dropout))
    layers.append(Dense(1, activation='sigmoid'))
    return Sequential(layers)

print('Funcion build_model definida')


In [ ]:
print('Fase 1: Busqueda de arquitectura')

results_p1 = []

for arch_name, arch_cfg in ARCHITECTURES.items():
    print(f'  Probando {arch_name}...')
    with mlflow.start_run(run_name=f'arch_{arch_name}'):
        model = build_model(arch_cfg, norm_layer, X_train.shape[1], use_bn=False)
        model.compile(optimizer='adam', loss='binary_crossentropy',
                     metrics=['accuracy', keras.metrics.AUC(name='auc')])
        early = EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)
        history = model.fit(X_train_arr, y_train, epochs=100, batch_size=64,
                           validation_split=0.2, class_weight=class_weight_dict,
                           callbacks=[early], verbose=0)
        y_pred_proba = model.predict(X_test_arr, verbose=0).flatten()
        y_pred = (y_pred_proba > 0.5).astype(int)
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        auc  = roc_auc_score(y_test, y_pred_proba)
        mlflow.log_param('architecture', arch_name)
        mlflow.log_param('layers', str(arch_cfg['layers']))
        mlflow.log_metric('accuracy', acc)
        mlflow.log_metric('precision', prec)
        mlflow.log_metric('recall', rec)
        mlflow.log_metric('f1_score', f1)
        mlflow.log_metric('auc_roc', auc)
        mlflow.keras.log_model(model, 'model')
        results_p1.append({'arch': arch_name, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc})
        print(f'    AUC: {auc:.4f}, Recall: {rec:.4f}')

df_p1 = pd.DataFrame(results_p1)
print(df_p1.sort_values('auc', ascending=False).to_string())


In [ ]:
best_arch = df_p1.loc[df_p1['auc'].idxmax(), 'arch']
print(f'Mejor arquitectura: {best_arch}')
print(f'  AUC: {df_p1.loc[df_p1["auc"].idxmax(), "auc"]:.4f}')


In [ ]:
print('Fase 2: Comparacion con BatchNormalization')

results_p2 = []

for use_bn in [False, True]:
    print(f'  BatchNorm={use_bn}...')
    with mlflow.start_run(run_name=f'bn_{use_bn}_{best_arch}'):
        model = build_model(ARCHITECTURES[best_arch], norm_layer, X_train.shape[1], use_bn=use_bn)
        model.compile(optimizer='adam', loss='binary_crossentropy',
                     metrics=['accuracy', keras.metrics.AUC(name='auc')])
        early = EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)
        history = model.fit(X_train_arr, y_train, epochs=100, batch_size=64,
                           validation_split=0.2, class_weight=class_weight_dict,
                           callbacks=[early], verbose=0)
        y_pred_proba = model.predict(X_test_arr, verbose=0).flatten()
        y_pred = (y_pred_proba > 0.5).astype(int)
        auc = roc_auc_score(y_test, y_pred_proba)
        rec = recall_score(y_test, y_pred)
        mlflow.log_param('batchnorm', use_bn)
        mlflow.log_metric('auc_roc', auc)
        mlflow.keras.log_model(model, 'model')
        results_p2.append({'bn': use_bn, 'auc': auc, 'rec': rec})
        print(f'    AUC: {auc:.4f}')

df_p2 = pd.DataFrame(results_p2)
print(df_p2.sort_values('auc', ascending=False).to_string())


In [ ]:
print('Fase 3: Ajuste de tasa de aprendizaje')

best_bn = df_p2.loc[df_p2['auc'].idxmax(), 'bn']
learning_rates = [0.0001, 0.0005, 0.001, 0.005]
results_p3 = []

for lr in learning_rates:
    print(f'  LR={lr}...')
    with mlflow.start_run(run_name=f'lr_{lr}_{best_arch}'):
        model = build_model(ARCHITECTURES[best_arch], norm_layer, X_train.shape[1], use_bn=best_bn)
        opt = keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=opt, loss='binary_crossentropy',
                     metrics=['accuracy', keras.metrics.AUC(name='auc')])
        early = EarlyStopping(monitor='val_auc', mode='max', patience=15, restore_best_weights=True)
        history = model.fit(X_train_arr, y_train, epochs=100, batch_size=64,
                           validation_split=0.2, class_weight=class_weight_dict,
                           callbacks=[early], verbose=0)
        y_pred_proba = model.predict(X_test_arr, verbose=0).flatten()
        auc = roc_auc_score(y_test, y_pred_proba)
        mlflow.log_param('learning_rate', lr)
        mlflow.log_metric('auc_roc', auc)
        mlflow.keras.log_model(model, 'model')
        results_p3.append({'lr': lr, 'auc': auc})
        print(f'    AUC: {auc:.4f}')

df_p3 = pd.DataFrame(results_p3)
print(df_p3.sort_values('auc', ascending=False).to_string())


In [ ]:
best_lr  = df_p3.loc[df_p3['auc'].idxmax(), 'lr']
best_auc = df_p3.loc[df_p3['auc'].idxmax(), 'auc']

print('Mejor configuracion encontrada:')
print(f'  Arquitectura:     {best_arch} {ARCHITECTURES[best_arch]["layers"]}')
print(f'  BatchNorm:        {best_bn}')
print(f'  Tasa aprendizaje: {best_lr}')
print(f'  AUC-ROC:          {best_auc:.4f}')


In [ ]:
print('Entrenando modelo final...')

with mlflow.start_run(run_name='MODELO_FINAL'):
    final_model = build_model(ARCHITECTURES[best_arch], norm_layer, X_train.shape[1], use_bn=best_bn)
    opt = keras.optimizers.Adam(learning_rate=best_lr)
    final_model.compile(optimizer=opt, loss='binary_crossentropy',
                       metrics=['accuracy', keras.metrics.AUC(name='auc'),
                               keras.metrics.Precision(name='precision'),
                               keras.metrics.Recall(name='recall')])
    early = EarlyStopping(monitor='val_auc', mode='max', patience=20, restore_best_weights=True)
    history_final = final_model.fit(
        X_train_arr, y_train,
        epochs=150,
        batch_size=64,
        validation_split=0.2,
        class_weight=class_weight_dict,
        callbacks=[early],
        verbose=1
    )
    print('Modelo final entrenado')


In [ ]:
# Evaluacion del modelo final
y_pred_proba_final = final_model.predict(X_test_arr, verbose=0).flatten()
y_pred_final = (y_pred_proba_final > 0.5).astype(int)

acc_f  = accuracy_score(y_test, y_pred_final)
prec_f = precision_score(y_test, y_pred_final, zero_division=0)
rec_f  = recall_score(y_test, y_pred_final)
f1_f   = f1_score(y_test, y_pred_final)
auc_f  = roc_auc_score(y_test, y_pred_proba_final)

print('Resultados - Modelo Final')
print(f'  Accuracy:  {acc_f:.4f} ({acc_f*100:.1f}%)')
print(f'  Precision: {prec_f:.4f} ({prec_f*100:.1f}%)')
print(f'  Recall:    {rec_f:.4f} ({rec_f*100:.1f}%)')
print(f'  F1-Score:  {f1_f:.4f}')
print(f'  AUC-ROC:   {auc_f:.4f}')

cm_f = confusion_matrix(y_test, y_pred_final)
print('\nMatriz de Confusion:')
print(f'  TN={cm_f[0,0]:5d}  FP={cm_f[0,1]:5d}')
print(f'  FN={cm_f[1,0]:5d}  TP={cm_f[1,1]:5d}')
print(f'\n{classification_report(y_test, y_pred_final, target_names=["No Bilingue", "Bilingue"])}')

# Registrar metricas finales en MLflow
mlflow.log_param('FINAL', True)
mlflow.log_metric('accuracy', acc_f)
mlflow.log_metric('auc_roc', auc_f)
mlflow.keras.log_model(final_model, 'final_model')


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Modelo Final - Analisis', fontsize=16, fontweight='bold')

# Matriz de confusion
ConfusionMatrixDisplay(cm_f, display_labels=['No Bilingue', 'Bilingue']).plot(ax=axes[0,0], cmap='Blues')
axes[0,0].set_title('Matriz de Confusion')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_final)
axes[0,1].plot(fpr, tpr, 'orange', lw=2, label=f'ROC (AUC={auc_f:.3f})')
axes[0,1].plot([0,1], [0,1], 'navy', lw=2, linestyle='--')
axes[0,1].set_title('Curva ROC')
axes[0,1].set_xlabel('Tasa de Falsos Positivos')
axes[0,1].set_ylabel('Tasa de Verdaderos Positivos')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Perdida de entrenamiento
axes[1,0].plot(history_final.history['loss'], label='Entrenamiento')
axes[1,0].plot(history_final.history['val_loss'], label='Validacion')
axes[1,0].set_title('Perdida')
axes[1,0].set_xlabel('Epocas')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# AUC de entrenamiento
axes[1,1].plot(history_final.history['auc'], label='Entrenamiento')
axes[1,1].plot(history_final.history['val_auc'], label='Validacion')
axes[1,1].set_title('AUC-ROC')
axes[1,1].set_xlabel('Epocas')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import os

# Guardar modelo y lista de variables en carpeta modelos/
os.makedirs('../modelos', exist_ok=True)
final_model.save('../modelos/mejor_modelo_real.keras')
print('Modelo guardado: modelos/mejor_modelo_real.keras')

with open('../modelos/feature_names_real.txt', 'w') as f:
    for feat in feature_cols:
        f.write(f'{feat}\n')
print('Variables guardadas: modelos/feature_names_real.txt')

print('\nResumen:')
print(f'  Registros procesados: {len(df_clean):,}')
print(f'  Variables:            {len(feature_cols)}')
print(f'  Arquitectura:         {best_arch}')
print(f'  BatchNorm:            {best_bn}')
print(f'  Tasa aprendizaje:     {best_lr}')
print(f'  AUC-ROC final:        {auc_f:.4f}')
print(f'  Recall final:         {rec_f:.4f}')


## Ver experimentos en MLflow

Para explorar los resultados de las corridas:

```bash
mlflow ui
```

Abrir: http://localhost:5000
